
<center>
<img src="../images/fscampus_small2.png" width="1200"/>
</center>

<center>

# Investments

***Finance 2 - BFIN***

**Dr. Omer Cayirli**

Lecturer in Empirical Finance

omer_cayirli@uncbusiness.net
</center>

---

## Lecture 03
---

### Outline
*   Fixed-Income Securities II
    *   Managing Bond Portfolios
        *   Interest rate risk
            *   Interest rate sensitivity of bond prices
            *   Duration and its determinants
            *   Convexity
        *   Passive and active management strategies

---



### Interest Rate Risk
*   Bond prices and yields move in opposite directions.

*   For equal changes in yield, a decrease produces a larger price gain than the loss caused by an increase.

*   Longer maturities and lower coupon rates increase price sensitivity.

Assume a USD 1,000 face value, semiannual compounding, and a 6% initial yield to maturity (APR). The coupon bond pays USD 30 every six months.

#### 6% Coupon Bond

| YTM (APR) | $T = 1$ | $T = 10$ | $T = 20$ |
|:---|---:|---:|---:|
| 5% | 1,009.64 | 1,077.95 | 1,125.51 |
| 6% (initial) | 1,000.00 | 1,000.00 | 1,000.00 |
| 7% | 990.50 | 928.94 | 893.22 |
| Price change: 6% to 5% | +0.96% | +7.79% | +12.55% |
| Price change: 6% to 7% | -0.95% | -7.11% | -10.68% |

#### Zero-Coupon Bond

| YTM (APR) | $T = 1$ | $T = 10$ | $T = 20$ |
|:---|---:|---:|---:|
| 5% | 951.81 | 610.27 | 372.43 |
| 6% (initial) | 942.60 | 553.68 | 306.56 |
| 7% | 933.51 | 502.57 | 252.57 |
| Price change: 6% to 5% | +0.98% | +10.22% | +21.49% |
| Price change: 6% to 7% | -0.96% | -9.23% | -17.61% |

---



In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# --- 1. Core Calculation Function ---
def calculate_bond_price(face_value, coupon_rate, ytm, years):
    coupon_payment = face_value * coupon_rate
    if ytm <= 0: return np.nan
    price = coupon_payment * ((1 - (1 + ytm)**-years) / ytm) + face_value / (1 + ytm)**years
    return price

# --- 2. Combined Plotting Function for All Three Bonds ---
def generate_linked_plots(c_a, t_a, c_b, t_b, c_c, t_c, initial_ytm):
    face_value = 1000.0
    ytm_r = initial_ytm / 100

    # Define the parameters for the three bonds based on widget inputs
    bonds = {
        'A': {'coupon': c_a/100, 'maturity': t_a, 'style': '-', 'label': f'A: {c_a:g}% coupon, {t_a} years'},
        'B': {'coupon': c_b/100, 'maturity': t_b, 'style': '--', 'label': f'B: {c_b:g}% coupon, {t_b} years'},
        'C': {'coupon': c_c/100, 'maturity': t_c, 'style': ':', 'label': f'C: {c_c:g}% coupon, {t_c} years'}
    }

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Interest-rate sensitivity ($1,000 face value; annual coupons and compounding)', fontsize=15)

    # --- Graph 1: YTM and Price of Bonds ---
    ytm_range = np.linspace(0.02, 0.20, 100)
    for key, params in bonds.items():
        prices = [calculate_bond_price(face_value, params['coupon'], y, params['maturity']) for y in ytm_range]
        ax1.plot(ytm_range * 100, prices, label=params['label'], linestyle=params['style'], linewidth=2)
    
    ax1.axvline(initial_ytm, color='0.4', linestyle='-.', linewidth=1, label='Initial YTM')
    ax1.set_title('Bond price versus YTM', fontsize=13)
    ax1.set_xlabel('Yield to maturity (%)', fontsize=11)
    ax1.set_ylabel('Bond price ($)', fontsize=11)
    ax1.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
    ax1.grid(True, linestyle=':', alpha=0.7)
    ax1.legend(fontsize=9)

    # --- Graph 2: Percentage Change in Bond Price ---
    ytm_change_range = np.linspace(-0.05, 0.05, 101)
    for key, params in bonds.items():
        initial_price = calculate_bond_price(face_value, params['coupon'], ytm_r, params['maturity'])
        pct_change = [((calculate_bond_price(face_value, params['coupon'], ytm_r + dy, params['maturity']) / initial_price) - 1) * 100 if initial_price else 0 for dy in ytm_change_range]
        ax2.plot(ytm_change_range * 100, pct_change, label=params['label'], linestyle=params['style'], linewidth=2)
    
    ax2.axhline(y=0, color='black', linewidth=0.75)
    ax2.axvline(x=0, color='black', linewidth=0.75)
    ax2.set_title(f'Price change from {initial_ytm:g}% initial YTM', fontsize=13)
    ax2.set_xlabel('Change in YTM (percentage points)', fontsize=11)
    ax2.set_ylabel('Bond price change (%)', fontsize=11)
    ax2.grid(True, linestyle=':', alpha=0.7)
    ax2.legend(fontsize=9)
    
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    plt.show()

# --- 3. Interactive Widgets Setup ---
output_widget = widgets.Output()
style = {'description_width': 'initial'}

# Widgets for Bond A
c_a_widget = widgets.FloatSlider(value=12.0, min=2, max=15, step=0.5, description='Coupon (%):', style=style, continuous_update=True)
t_a_widget = widgets.IntSlider(value=10, min=1, max=30, step=1, description='Maturity (Y):', style=style, continuous_update=True)

# Widgets for Bond B
c_b_widget = widgets.FloatSlider(value=10.0, min=2, max=15, step=0.5, description='Coupon (%):', style=style, continuous_update=True)
t_b_widget = widgets.IntSlider(value=20, min=1, max=30, step=1, description='Maturity (Y):', style=style, continuous_update=True)

# Widgets for Bond C
c_c_widget = widgets.FloatSlider(value=8.0, min=2, max=15, step=0.5, description='Coupon (%):', style=style, continuous_update=True)
t_c_widget = widgets.IntSlider(value=30, min=1, max=30, step=1, description='Maturity (Y):', style=style, continuous_update=True)

# Single slider for the shared Initial YTM
initial_ytm_widget = widgets.FloatSlider(value=10.0, min=2, max=15, step=0.5, description='Initial YTM (%):', style=style, continuous_update=True)

# Define the handler that updates everything
def interactive_handler(c_a, t_a, c_b, t_b, c_c, t_c, initial_ytm):
    with output_widget:
        clear_output(wait=True)
        generate_linked_plots(c_a, t_a, c_b, t_b, c_c, t_c, initial_ytm)

# Organize UI
bond_controls = widgets.HBox([
    widgets.VBox([widgets.HTML("<h3>Bond A</h3>"), c_a_widget, t_a_widget]),
    widgets.VBox([widgets.HTML("<h3>Bond B</h3>"), c_b_widget, t_b_widget]),
    widgets.VBox([widgets.HTML("<h3>Bond C</h3>"), c_c_widget, t_c_widget])
])
ui = widgets.VBox([bond_controls, initial_ytm_widget])


# Link all widgets to the handler
widgets.interactive_output(interactive_handler, {
    'c_a': c_a_widget, 't_a': t_a_widget,
    'c_b': c_b_widget, 't_b': t_b_widget,
    'c_c': c_c_widget, 't_c': t_c_widget,
    'initial_ytm': initial_ytm_widget
})

# Display UI and the output area
display(ui, output_widget)
interactive_handler(c_a_widget.value, t_a_widget.value, c_b_widget.value, t_b_widget.value, c_c_widget.value, t_c_widget.value, initial_ytm_widget.value)

Output()

---

### Duration
*   A measure of the effective maturity of a bond
*   The weighted average of the times until each payment is received
*   The weights are proportional to the present value of the payment
    *   Duration = Maturity for zero coupon bonds
    *   Duration < Maturity for coupon bonds

$$D = \sum_{t=1}^{T} t \times w_t \qquad \text{where} \qquad w_t = \frac{CF_t/((1+y)^t)}{P}$$

*   Duration is a key concept in fixed-income portfolio management.
    *   A simple summary statistic of the effective average maturity of the portfolio.
    *   An essential tool in immunizing portfolios from interest rate risk.
    *   A measure of the interest rate sensitivity of a portfolio.

*$CF_t$ is the cash flow at time t; $P$ is the price of the bond, and $y$ is the yield to maturity (YTM).*

---



### Duration-Price Relationship
*   Price change is proportional to duration.
*   The percentage change in bond price is the product of modified duration and the change in the bond's yield to maturity.

$$\frac{\Delta P}{P} = -D \times \frac{\Delta y}{1+y}$$

$$\frac{\Delta P}{P} = -D_M \Delta y \qquad \text{where} \qquad D_M = \frac{D}{1 + y} \qquad \text{and} \qquad D_M \quad \text{is modified duration}$$

*   Rule 1: The duration of a zero-coupon bond equals its time to maturity.

*   Rule 2: Holding maturity constant, a bond's duration is higher when the coupon rate is lower.

*   Rule 3: Holding the coupon rate constant, a bond's duration generally increases with its time to maturity.

*   Rule 4: Holding other factors constant, the duration of a coupon bond is higher when the bond's yield to maturity is lower.

*   Rule 5: The duration of a level perpetuity is equal to: $\frac{1+y}{y}$

---

In [2]:
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# --- 1. Core Calculation Logic ---
def get_bond_cashflows(face_value, coupon_rate, ytm, years, freq=1):
    # Convert to decimal and period rates
    c = coupon_rate / 100
    y = ytm / 100
    period_coupon = (face_value * c) / freq
    period_ytm = y / freq
    n_periods = int(years * freq)
    
    times = np.arange(1, n_periods + 1) / freq
    cfs = np.full(n_periods, period_coupon)
    cfs[-1] += face_value # Add face value to last payment
    
    pvs = cfs / (1 + period_ytm)**np.arange(1, n_periods + 1)
    price = np.sum(pvs)
    weights = pvs / price
    weighted_times = times * weights
    
    macaulay_duration = np.sum(weighted_times)
    modified_duration = macaulay_duration / (1 + period_ytm)
    
    return {
        'times': times,
        'pvs': pvs,
        'weights': weights,
        'weighted_times': weighted_times,
        'price': price,
        'D': macaulay_duration,
        'Dm': modified_duration,
        'freq': freq,
        'years': years
    }

# --- 2. HTML Table Generation ---
def generate_html_table(data, title):
    rows_html = ""
    # Generate rows for each period
    for t, pv, w, tw in zip(data['times'], data['pvs'], data['weights'], data['weighted_times']):
        rows_html += f"""
        <tr>
            <td style="text-align: center;">{t:.1f}</td>
            <td style="text-align: right;">{pv:,.2f}</td>
            <td style="text-align: center;">{w:.4f}</td>
            <td style="text-align: center;">{tw:.4f}</td>
        </tr>
        """
    
    # Construct the full table with summary footer
    table_html = f"""
    <div style="flex: 1; min-width: 300px; margin: 5px; border: 1px solid #ddd; padding: 10px; border-radius: 5px;">
        <h4 style="text-align: center; margin-top: 0;">{title}</h4>
        <table style="width: 100%; border-collapse: collapse; font-family: monospace; font-size: 0.9em;">
            <thead style="background-color: #f9f9f9; border-bottom: 2px solid #aaa;">
                <tr>
                    <th style="text-align: center;">t</th>
                    <th style="text-align: right;">PV(CF)</th>
                    <th style="text-align: center;">w<sub>t</sub></th>
                    <th style="text-align: center;">t*w<sub>t</sub></th>
                </tr>
            </thead>
            <tbody>
                {rows_html}
            </tbody>
            <tfoot style="border-top: 2px solid #aaa; font-weight: bold;">
                <tr>
                    <td style="text-align: center;">Price</td>
                    <td style="text-align: right;">{data['price']:,.2f}</td>
                    <td style="text-align: center;">1.00</td>
                    <td></td>
                </tr>
                <tr>
                    <td colspan="2"></td>
                    <td style="text-align: center; background-color: #eef;">D</td>
                    <td style="text-align: center; background-color: #eef;">{data['D']:.4f}</td>
                </tr>
                <tr>
                    <td colspan="2"></td>
                    <td style="text-align: center; background-color: #eef;">D<sub>M</sub></td>
                    <td style="text-align: center; background-color: #eef;">{data['Dm']:.4f}</td>
                </tr>
            </tfoot>
        </table>
    </div>
    """
    return table_html

# --- 3. Interactive Handler & UI ---
output_widget = widgets.Output()
style = {'description_width': 'initial'}

face_widget = widgets.FloatText(value=10000, description='Face Value:', style=style)
coupon_widget = widgets.FloatSlider(value=12.0, min=1, max=20, step=0.5, description='Coupon (%):', style=style, continuous_update=True)
maturity_widget = widgets.IntSlider(value=5, min=2, max=15, step=1, description='Maturity (Y):', style=style, continuous_update=True)
ytm_widget = widgets.FloatSlider(value=12.0, min=1, max=20, step=0.5, description='YTM (%):', style=style, continuous_update=True)

def update_tables(face, coupon, maturity, ytm):
    # Calculate data for all three bonds
    b1 = get_bond_cashflows(face, coupon, ytm, maturity, freq=1)
    b2 = get_bond_cashflows(face, coupon, ytm, maturity, freq=2)
    b3 = get_bond_cashflows(face, coupon, ytm, maturity * 2, freq=1)
    
    # Generate HTML for each
    html1 = generate_html_table(b1, f"Bond 1 (Annual, {maturity}Y)")
    html2 = generate_html_table(b2, f"Bond 2 (Semi-Annual, {maturity}Y)")
    html3 = generate_html_table(b3, f"Bond 3 (Annual, {maturity*2}Y)")
    
    # Combine into one flexbox container
    full_html = f"""
    <div style="display: flex; flex-wrap: wrap; justify-content: center; width: 100%;">
        {html1}
        {html2}
        {html3}
    </div>
    """
    
    with output_widget:
        clear_output(wait=True)
        display(HTML(full_html))

# Link widgets
widgets.interactive_output(update_tables, {
    'face': face_widget,
    'coupon': coupon_widget,
    'maturity': maturity_widget,
    'ytm': ytm_widget
})

# Layout
controls = widgets.HBox([
    widgets.VBox([face_widget, coupon_widget]),
    widgets.VBox([maturity_widget, ytm_widget])
])

display(controls, output_widget)

# Initial load
update_tables(face_widget.value, coupon_widget.value, maturity_widget.value, ytm_widget.value)

Output()

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# --- 1. Core Calculation Function (with fixes) ---
def calculate_macaulay_duration(coupon_rate, ytm, maturity, face_value=1000, freq=1):
    # This function calculates the Macaulay Duration for a single bond.
    if maturity == 0:
        return 0
    if coupon_rate == 0:
        return maturity
        
    c = coupon_rate / 100
    y = ytm / 100
    
    # Guard against division by zero or near-zero yields.
    if abs(y) < 1e-9:
        y = 1e-9
    
    periods = maturity
    
    # price = (c * face_value / y) * (1 - 1/((1+y)**periods)) + face_value/((1+y)**periods)
    
    # Macaulay Duration formula (closed-form)
    duration = ((1 + y) / y) - ( (1 + y) + periods * (c - y) ) / ( c * ((1+y)**periods - 1) + y )
    return duration

# --- 2. Plotting Function (with fixes) ---
def generate_duration_plot(ytm, c1, c2, c3):
    fig, ax = plt.subplots(figsize=(10, 6))
    
    maturities = np.arange(1, 31)

    # 1. Plot the Zero-Coupon Bond
    ax.plot(maturities, maturities, color='black', label='Zero-Coupon Bond')

    # 2. Define the three bonds
    bonds = [
        {'coupon': c1, 'color': 'deepskyblue', 'style': '-'},
        {'coupon': c2, 'color': 'gray', 'style': '-'},
        {'coupon': c3, 'color': 'dodgerblue', 'style': '--'}
    ]

    # 3. Calculate and plot duration for each bond
    for bond in bonds:
        durations = [calculate_macaulay_duration(bond['coupon'], ytm, m) for m in maturities]
        ax.plot(maturities, durations, color=bond['color'], linestyle=bond['style'], 
                label=f'{bond["coupon"]}% Coupon', linewidth=2.5)

    # --- Formatting with Fix 5 for clarity and the requested title format ---
    ax.set_title(f'Duration vs. Maturity (Annual Payments, Global YTM = {ytm:.2f}%)', fontsize=15)
    ax.set_xlabel('Maturity (years)', fontsize=12)
    ax.set_ylabel('Duration (years)', fontsize=12)
    ax.set_xlim(0, 30)
    ax.set_ylim(0, 30)
    ax.legend()
    ax.grid(True, linestyle=':', alpha=0.7)
    
    plt.show()

# --- 3. Interactive Widgets Setup (with fixes) ---
output_widget = widgets.Output()
style = {'description_width': 'initial'}

ytm_widget = widgets.FloatSlider(value=10.0, min=1, max=20, step=0.5, description='Global YTM (%):', style=style, continuous_update=True)
c1_widget = widgets.FloatSlider(value=12.0, min=0, max=20, step=0.5, description='Bond 1 Coupon (%):', style=style, continuous_update=True)
c2_widget = widgets.FloatSlider(value=8.0, min=0, max=20, step=0.5, description='Bond 2 Coupon (%):', style=style, continuous_update=True)
c3_widget = widgets.FloatSlider(value=10.0, min=0, max=20, step=0.5, description='Bond 3 Coupon (%):', style=style, continuous_update=True)

def interactive_handler(ytm, c1, c2, c3):
    with output_widget:
        clear_output(wait=True)
        generate_duration_plot(ytm, c1, c2, c3)

# Fix 1: Keep a reference to the interactive_output widget to prevent garbage collection.
_io_link = widgets.interactive_output(interactive_handler, {
    'ytm': ytm_widget,
    'c1': c1_widget,
    'c2': c2_widget,
    'c3': c3_widget
})

# Organize the UI
bond_controls = widgets.VBox([c1_widget, c2_widget, c3_widget])
ui = widgets.VBox([ytm_widget, bond_controls])

# Display the UI and the output area
display(ui, output_widget)
interactive_handler(ytm_widget.value, c1_widget.value, c2_widget.value, c3_widget.value)

Output()

---

### Interest Rate Risk

* Modified duration measures the sensitivity of a bond's price to a small change in its annual yield $y$.

$$
P = \sum_{k=1}^{mT}\frac{CF_k}{(1+y/m)^k},
\qquad D_M=-\frac{1}{P}\frac{dP}{dy},
\qquad \Delta P\approx-PD_M\Delta y
$$

Here $m$ is the number of coupon payments per year and $T$ is maturity in years.

* Money (dollar) duration is $PD_M=-dP/dy$. 
    * For a portfolio, money durations add when yields move together.

* DV01 is the value of a one-basis-point yield change: 
    *   $\text{DV01}\approx PD_M\times0.0001$. 
    *   It is the approximate price decrease for a one-basis-point rise in yield; portfolio DV01s also add.

---

### Interest Rate Risk

A bond with a par value of €1,000, a maturity of 6 years, and semi-annual coupon payments has a price of €1,075.78 and a yield to maturity of 5.5%.

* What is the annual coupon rate?
* What is the Macaulay duration of the bond at the current price?
* What is the modified duration?
* What is the DV01 at the current yield?
* Estimate the price change given a 50-basis-point decline in the YTM.
* What is the actual price of the bond when the YTM declines to 5.0%?

---

### Interest Rate Risk

$$
\begin{aligned}
1,075.78 &= C\sum_{k=1}^{12}\frac{1}{(1+2.75\%)^k}+\frac{1,000}{(1+2.75\%)^{12}} \\
&\approx 10.1042C+722.13
\end{aligned}
$$

$$
C\approx\frac{1,075.78-722.13}{10.1042}=35.00
\qquad\Rightarrow\qquad
\text{Annual coupon rate}=\frac{2\times35}{1,000}=7\%
$$

The present-value weights in the table give Macaulay duration $D=5.0429$ years. Therefore,

$$
D_M=\frac{D}{1+y/2}=\frac{5.0429}{1.0275}=4.9080\text{ years}
$$

$$
\text{DV01}\approx PD_M(0.0001)=1,075.78\times 4.9080\times0.0001=0.5280\text{ per bp}
$$

For a decline of 50 bp, $\Delta y=-0.005$:

$$
\Delta P\approx-PD_M\Delta y=+26.40
\quad\text{or}\quad
\frac{\Delta P}{P}\approx +2.454\%.
$$

At the new yield,

$$ P(5.0\%)=\sum_{k=1}^{12}\frac{35}{(1+2.50\%)^k}+\frac{1,000}{(1+2.50\%)^{12}}=1,102.58.$$

The actual price change is $+26.80$, or $+2.49\%$.

| t | PV(CF)@5.5% | w<sub>t</sub>@5.5% | t\*w<sub>t</sub>@5.5% | PV(CF)@5.0% | w<sub>t</sub>@5.0% | t\*w<sub>t</sub>@5.0% |
|---|---|---|---|---|---|---|
| 0.5 | 34.06 | 0.0317 | 0.0158 | 34.15 | 0.0310 | 0.0155 |
| 1.0 | 33.15 | 0.0308 | 0.0308 | 33.31 | 0.0302 | 0.0302 |
| 1.5 | 32.26 | 0.0300 | 0.0450 | 32.50 | 0.0295 | 0.0442 |
| 2.0 | 31.40 | 0.0292 | 0.0584 | 31.71 | 0.0288 | 0.0575 |
| 2.5 | 30.56 | 0.0284 | 0.0710 | 30.93 | 0.0281 | 0.0701 |
| 3.0 | 29.74 | 0.0276 | 0.0829 | 30.18 | 0.0274 | 0.0821 |
| 3.5 | 28.95 | 0.0269 | 0.0942 | 29.44 | 0.0267 | 0.0935 |
| 4.0 | 28.17 | 0.0262 | 0.1047 | 28.73 | 0.0261 | 0.1042 |
| 4.5 | 27.42 | 0.0255 | 0.1147 | 28.03 | 0.0254 | 0.1144 |
| 5.0 | 26.68 | 0.0248 | 0.1240 | 27.34 | 0.0248 | 0.1240 |
| 5.5 | 25.97 | 0.0241 | 0.1328 | 26.68 | 0.0242 | 0.1331 |
| 6.0 | 747.41 | 0.6948 | 4.1686 | 769.58 | 0.6980 | 4.1879 |
| Totals: | 1,075.78 | 1.0000 | 5.0429 | 1,102.58 | 1.0000 | 5.0567 |

---

### Interest Rate Risk: Convexity

* Modified duration gives a straight-line approximation to the price change for a small change in annual YTM.
* Convexity measures the curvature of the price-yield relationship. For an option-free bond with positive convexity, the actual price is above the duration-only estimate for both a yield rise and a yield fall.

Let $Y$ be annual YTM, $f$ the number of payments per year, and $k$ the payment period. With $CF_k$ including principal in the last period,

$$
P(Y)=\sum_{k=1}^{fT}\frac{CF_k}{(1+Y/f)^k}
$$

$$
\text{Annual convexity}
=\frac{1}{P f^2(1+Y/f)^2}
\sum_{k=1}^{fT}k(k+1)\frac{CF_k}{(1+Y/f)^k}
$$

$$
\frac{\Delta P}{P}\approx-D_M\Delta Y
+\frac{1}{2}\text{Annual convexity}(\Delta Y)^2
$$

---

### Interest Rate Risk: Convexity

Consider a 5-year bond with a par value of €1,000, semi-annual coupons, an annual coupon rate of 6%, and a current YTM of 6.5%.

* What are its current price, Macaulay duration, and modified duration?
* What is its annual convexity?
* If YTM falls by 100 basis points to 5.5%, estimate the price change using (a) duration alone and (b) duration plus convexity. What is the actual new price?
* Repeat for a 100-basis-point rise in YTM to 7.5%.

---

### Interest Rate Risk: Convexity — Solution

The semi-annual coupon is €30. Amounts in the calculations below are in euros, and intermediate values are unrounded.

$$
P(Y)=\sum_{k=1}^{10}\frac{30}{(1+Y/2)^k}+\frac{1000}{(1+Y/2)^{10}},
\qquad P_0=P(0.065)=978.94
$$

The present-value-weighted cash-flow times give

$$
D=\frac{4292.9306}{978.9440}=4.3853\text{ years},
\qquad D_M=\frac{D}{1.0325}=4.2472\text{ years}
$$

Using the annual-YTM convention from the previous slide,

$$
\text{Annual convexity}
=\frac{90215.6396}{4(978.9440)(1.0325)^2}=21.6114
$$

For either yield move, use $\Delta P/P_0\approx-D_M\Delta Y+\tfrac12(21.6114)(\Delta Y)^2$. Repricing with $P(Y)$ gives the actual result.

| New YTM | $\Delta Y$ | Duration-only $\Delta P$ | Duration + convexity $\Delta P$ | Actual price | Actual $\Delta P$ |
|---|---:|---:|---:|---:|---:|
| 5.5% | −0.01 | +41.58 | +42.64 | 1,021.60 | +42.66 |
| 7.5% | +0.01 | −41.58 | −40.52 | 938.40 | −40.54 |

The convexity adjustment reduces the price-change error from about €1.08 to €0.02 for the yield fall, and from about €1.04 to €0.02 for the yield rise.

---

### Interest Rate Risk: Convexity

Convexity can also be estimated from three prices around the current annual YTM $Y$:

$$
C_{\mathrm{FD}}\approx
\frac{P(Y-h)+P(Y+h)-2P(Y)}{P(Y)h^2},
\qquad h=\text{change in annual YTM}
$$

Consider a 7-year bond with a par value of €10,000, semi-annual coupons, an annual coupon rate of 5%, and a current YTM of 5%. Find its prices at YTMs of 4%, 5%, and 6%, then estimate its annual convexity.

The semi-annual coupon is €250. All prices below are in euros.

$$
P(Y)=\sum_{k=1}^{14}\frac{250}{(1+Y/2)^k}
+\frac{10000}{(1+Y/2)^{14}}
$$

$$
P(0.04)=10605.31,\qquad P(0.05)=10000.00,\qquad P(0.06)=9435.20
$$

A 1-percentage-point move in annual YTM means $h=0.01$, even though coupons are paid semi-annually. Using the displayed prices,

$$
C_{\mathrm{FD}}\approx
\frac{10605.31+9435.20-2(10000.00)}{10000.00(0.01)^2}
=40.51
$$

---

### Interest Rate Risk: Convexity

* **Higher Convexity** $\rightarrow$ Bigger price increases when yields fall than the price declines when yields rise.

* The more **volatile interest rates**, the more attractive this asymmetry.

* Bonds with greater **convexity** $\rightarrow$ higher prices and/or lower yields, all else equal.

---

### Interest Rate Risk: Price-Yield Curve for a Callable Bond

<img src="../images/slide_3/pic_7.png">

*   As rates fall, a call becomes more likely and limits further price gains. The bond can still trade above the call price before the call date.
*   This creates a region of negative convexity.

---



### Passive Management
*   Two passive bond portfolio strategies:
    *   Indexing
        *   Attempts to replicate the performance of a given bond index.
    *   Immunization
        *   Used widely by financial institutions such as insurance companies and pension funds to shield overall financial status from exposure to interest rate fluctuations.
*   Differ greatly in terms of risk
    *   A bond-index portfolio will have the similar risk-reward profile as the bond market index to which it is tied.
    *   Immunization strategies reduce the effect of small, parallel yield changes on a funded liability.
        *   Duration matching is approximate and requires rebalancing over time.

---



### Passive Management: Indexing

*   The idea is to create a portfolio that mirrors the composition of an index that measures the broad market.
    *   Broad market indexes
        *   Government, Agencies, Supras
        *   Corporate
        *   Mortgage-backed securities
        *   Yankee bonds
*   Challenges in constructing an indexed bond portfolio
    *   Indexes include thousands of securities
    *   Many bonds are very thinly traded
    *   Rebalancing problems
        *   Bonds are continually dropped from the index as they approach maturity.
        *   New bonds are added to the index as they are issued.
    *   Bonds generate considerable interest income that must be reinvested.

---



### Passive Management: Immunization
*   Control interest rate risk
*   Widely used by pension funds, insurance companies, and banks
*   A natural mismatch between asset and liability maturity structures.
    *   Bank liabilities are primarily the deposits owed to customers, most of which are short-term and, consequently, have low duration.
    *   Bank assets by contrast are composed largely of outstanding commercial and consumer loans or mortgages, which have longer duration.
    *   What happens when interest rates rise unexpectedly?
    *   What about the risks pension funds are exposed to?
*   The interest rate exposure of assets and liabilities is matched in the portfolio.
    *   Match the duration of the assets and liabilities.
    *   Price risk and reinvestment rate risk approximately offset near the matching point.
        *   With equal present values and matched dollar durations, assets approximately fund the liability after a small, parallel yield shift.
    *   Asset and liability values do not stay matched for every yield movement.
    *   Rebalancing is needed as time passes and yields change.

---



### Passive Management: Immunization 

*   Duration matching balances the difference between the accumulated value of the coupon payments (reinvestment rate risk) and the sale value of the bond (price risk).
    *   When interest rates fall, the coupons grow less than in the base case, but the higher value of the bond offsets this.
    *   When interest rates rise, the value of the bond falls, but the coupons approximately offset it because they are reinvested at the higher rate.

*   Coupon paying bond and single-payment obligation.
    *   For a small, parallel yield shift, equally funded assets and liabilities with matched durations change in value by approximately the same amount.
    *   For greater changes in the interest rate, the present value curves diverge.
    *   Convexity
    *   Rebalancing
        *   Interest rate changes cause mismatch.
        *   Asset durations will change with time.
        *   Without rebalancing, durations will become unmatched.

---

### Passive Management: Cash Flow Matching and Dedication

* Cash-flow matching constructs a bond portfolio whose cash flows equal the liability payments on the same dates.
    * A single known liability can be matched with a zero-coupon bond having the same maturity and payoff.
* Dedication extends cash-flow matching to a series of liabilities using zero-coupon and/or coupon bonds.
* An exact match removes interest-rate risk from funding those liabilities and requires no rebalancing for rate changes.
* The match can fail if promised bond payments are not made, embedded options alter cash flows, or the liabilities change.
* Exact matching may be difficult or costly when suitable bonds are unavailable or illiquid.

---

### Question 3.1

A pension fund must meet a single known liability in 7.5 years. Assume the bonds are default-free, noncallable, and denominated in the same currency as the liability. Compare these strategies:

1. Buy a zero-coupon bond maturing in 7.5 years with a payoff equal to the liability.
2. Build a coupon-bond portfolio whose present value equals the liability's present value and whose Macaulay duration is 7.5 years.

How do the strategies differ in reinvestment risk and the need for rebalancing?

---

### Question 3.2

What does positive convexity mean for an investor in an option-free bond?

---

### Question 3.3

A 4-year bond with a par value of €1,000 pays a 5.5% annual coupon semi-annually. Its current YTM is 7%.

* Calculate its price, Macaulay duration, modified duration, and annual convexity.
* If YTM rises by 75 basis points to 7.75%, estimate the new price using:
    1. modified duration;
    2. modified duration and convexity.
* Reprice the bond at 7.75% and compare the two estimates with the actual price.

---

### Question 3.4

A 10-year bond with a par value of €1,000 pays a 4.5% annual coupon once per year. Its current YTM is 6.5%, modified duration is 7.6102 years, and annual convexity is 72.9689 years². YTM falls by 100 basis points to 5.5%.

* Calculate the initial price and the actual price at the new YTM.
* Estimate the new price using modified duration only.
* Estimate the new price using modified duration and convexity.
* Calculate the absolute pricing error of each estimate. Which is more accurate?

---

### Question 3.5

A 15-year zero-coupon bond has a face value of €1,000, a YTM of 7%, and annual compounding. Its YTM falls by 150 basis points to 5.5%.

* Calculate the initial price, modified duration, and annual convexity.
* Estimate the percentage price change and new price using modified duration only.
* Repeat using modified duration and convexity.
* Calculate the actual new price and percentage change. Which estimate is more accurate?

---

### Question 3.6

A 6-year bond with a par value of €1,000 pays a 4% annual coupon semi-annually. Its current YTM is 5.5%.

* Calculate the bond prices at YTMs of 5.0%, 5.5%, and 6.0%.
* Use these prices to estimate annual convexity with a 50-basis-point change in YTM.

---

### Question 3.7

A pension fund must pay a €1,000,000 liability exactly 6 years from today. The yield curve is flat at 5% with annual compounding. The fund can invest in default-free zero-coupon bonds maturing in 4 and 10 years. Assume yield-curve changes are parallel.

* Calculate the current present value of the liability.
* Determine the current amount invested in each bond so that the asset portfolio has the same present value and Macaulay duration as the liability.
* Calculate the face value purchased of each bond.
* Immediately after a parallel yield increase to 6%, compare the asset value with the liability's present value. Is the position still fully funded?

---

### What is next?
*   Portfolio Theory and Practice I
    *   Risk, Return, and the Historical Record
    *   Capital Allocation to Risky Assets
        *   Readings: Ch. 5 & 6
    *   Suggested Problems
        *   Ch. 16: 4, 5, 9, 11, 12, 16, 21, 23
        *   Ch 16 – CFA Problems: 7, 12

---
